In [1]:
print("test working...")

test working...


In [2]:
%pwd

'd:\\Me\\Ai\\Projects\\Chatbot\\Cybersecurity-Risk-Advisor-Chatbot-with-LLMs-LangChain-Pinecone-Flask-AWS\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'd:\\Me\\Ai\\Projects\\Chatbot\\Cybersecurity-Risk-Advisor-Chatbot-with-LLMs-LangChain-Pinecone-Flask-AWS'

In [5]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\Users\Mahmoud Ibrahim\.conda\envs\medibot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [7]:
extracted_data = load_pdf_files("data")

In [8]:
extracted_data

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 16.2 (Macintosh)', 'creationdate': '2021-05-17T11:20:59-04:00', 'moddate': '2021-05-17T11:21:06-04:00', 'trapped': '/False', 'source': 'data\\CIS_Controls_v8_Guide.pdf', 'total_pages': 93, 'page': 0, 'page_label': '1'}, page_content='Page 1 \nCIS Controls v8\nCIS Controls\nVersion 8\nv8'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 16.2 (Macintosh)', 'creationdate': '2021-05-17T11:20:59-04:00', 'moddate': '2021-05-17T11:21:06-04:00', 'trapped': '/False', 'source': 'data\\CIS_Controls_v8_Guide.pdf', 'total_pages': 93, 'page': 1, 'page_label': '2'}, page_content='Page 2 \nCIS Controls v8\nCIS Controls Version 8\nAcknowledgments\nCIS would like to thank the many security experts who volunteer their time and talent to support the CIS \nControls and other CIS work. CIS products represent the effort of a veritable army of volunteers from \nacross the industry, genero

In [9]:
len(extracted_data)

378

## Clean Data

In [11]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [12]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [13]:
minimal_docs

[Document(metadata={'source': 'data\\CIS_Controls_v8_Guide.pdf'}, page_content='Page 1 \nCIS Controls v8\nCIS Controls\nVersion 8\nv8'),
 Document(metadata={'source': 'data\\CIS_Controls_v8_Guide.pdf'}, page_content='Page 2 \nCIS Controls v8\nCIS Controls Version 8\nAcknowledgments\nCIS would like to thank the many security experts who volunteer their time and talent to support the CIS \nControls and other CIS work. CIS products represent the effort of a veritable army of volunteers from \nacross the industry, generously giving their time and talent in the name of a more secure online experience \nfor everyone.\nCreative Commons License\nThis work is licensed under a\xa0Creative Commons Attribution-NonCommercial-No Derivatives 4.0 International \nPublic License (the link can be found at https://creativecommons.org/licenses/by-nc-nd/4.0/legalcode.\nTo further clarify the Creative Commons license related to the CIS Controls® content, you are authorized to \ncopy and redistribute the cont

In [14]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [15]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 1195


In [16]:
texts_chunk

[Document(metadata={'source': 'data\\CIS_Controls_v8_Guide.pdf'}, page_content='Page 1 \nCIS Controls v8\nCIS Controls\nVersion 8\nv8'),
 Document(metadata={'source': 'data\\CIS_Controls_v8_Guide.pdf'}, page_content='Page 2 \nCIS Controls v8\nCIS Controls Version 8\nAcknowledgments\nCIS would like to thank the many security experts who volunteer their time and talent to support the CIS \nControls and other CIS work. CIS products represent the effort of a veritable army of volunteers from \nacross the industry, generously giving their time and talent in the name of a more secure online experience \nfor everyone.\nCreative Commons License\nThis work is licensed under a\xa0Creative Commons Attribution-NonCommercial-No Derivatives 4.0 International \nPublic License (the link can be found at https://creativecommons.org/licenses/by-nc-nd/4.0/legalcode.\nTo further clarify the Creative Commons license related to the CIS Controls® content, you are authorized to \ncopy and redistribute the cont

In [17]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

C:\Users\Mahmoud Ibrahim\AppData\Local\Temp\ipykernel_12780\30168171.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\Mahmoud Ibrahim\.conda\envs\medibot\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mahmoud Ibrahim\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISAB

In [18]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [23]:
vector = embedding.embed_query("Expert Systems")
vector

[-0.018928444012999535,
 -0.005111827049404383,
 -0.008394732140004635,
 -0.009824623353779316,
 -0.018804872408509254,
 -0.03821340948343277,
 0.05892537534236908,
 0.07383397221565247,
 -0.05365367606282234,
 0.03380727395415306,
 -0.044377196580171585,
 -0.008568228222429752,
 0.14669117331504822,
 0.05216914787888527,
 -0.1192035973072052,
 0.03890743479132652,
 -0.016958007588982582,
 0.00323276175186038,
 -0.10907920449972153,
 -0.09970112890005112,
 -0.06577416509389877,
 -0.049764152616262436,
 -0.054580703377723694,
 0.006742747034877539,
 -0.02400914393365383,
 0.024587145075201988,
 0.03289363533258438,
 -0.01527484506368637,
 -0.006948279216885567,
 -0.055985018610954285,
 -0.0022846111096441746,
 -0.013818414881825447,
 0.05886565521359444,
 -0.040441013872623444,
 -0.03332868963479996,
 0.015875449404120445,
 -0.02927321009337902,
 0.07525509595870972,
 0.017975028604269028,
 0.027774494141340256,
 -0.05525367707014084,
 -0.03554489091038704,
 0.025085343047976494,
 0.015

In [24]:
print( "Vector length:", len(vector))

Vector length: 384


In [80]:
import os
from dotenv import load_dotenv

# This looks for a .env file in the current directory
load_dotenv() 

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if PINECONE_API_KEY is None or OPENAI_API_KEY is None:
    raise ValueError("API Keys not found. Check your .env file!")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [51]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [52]:
from pinecone import Pinecone 
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [53]:
pc

In [55]:
from pinecone import ServerlessSpec 

index_name = "cybersecuritychatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [56]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [57]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

# Add more data to the existing Pinecone index

In [58]:
info = Document(
    page_content="This project was implemented as an application of the Expert Systems course under the supervision of Dr. Reham El-Anany and Eng. Youssef. It was developed by students from the Computer and Control Engineering Department at the Faculty of Engineering.",
    metadata={"source": "Expert Systems"}
)

In [59]:
docsearch.add_documents(documents=[info])

['5bbe6dc8-0c58-4220-ae24-1301053e051a']

In [60]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [63]:
retrieved_docs = retriever.invoke("Who are the developers of this project?")
retrieved_docs

[Document(id='72b325b1-c3b3-4e3c-b1b1-fc6165ba1651', metadata={'source': 'data\\CIS_Controls_v8_Guide.pdf'}, page_content='software development best practices. The enterprise is attentive to the quality and \nmaintenance of third-party open source or commercial code on which it depends.\nDevelopment Group 3\n• The enterprise makes a major investment in custom software that it requires to run \nits business and serve its customers. It may host software on its own infrastructure, \nin the cloud, or both, and may integrate a large range of third-party open source and \ncommercial software components. Software vendors and enterprises that deliver \nSaaS should consider Development Group 3 as a minimum set of requirements.\nPage 52 \nCIS Controls v8Control 16: Application Software Security'),
 Document(id='6418680f-66ae-471c-8245-02b7e848c7b8', metadata={'source': 'data\\CIS_Controls_v8_Guide.pdf'}, page_content='The first step in developing an application security program is implementing a

In [97]:
!pip install "langchain-core<1.0.0" "langchain>=0.3.0,<1.0.0" "langchain-community>=0.3.0,<1.0.0" "langchain-pinecone>=0.2.0" "langchain-google-genai" --upgrade

  Using cached langchain_core-0.3.85-py3-none-any.whl.metadata (3.2 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_google_genai-4.2.2-py3-none-any.whl.metadata (2.7 kB)
INFO: pip is still looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + 

In [103]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_pinecone import PineconeVectorStore

chatModel = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

In [104]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [105]:
system_prompt = (
    "You are an Cybersecurity risk advisor for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [106]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [109]:
response = rag_chain.invoke({"input": "This chatbot application is for which subject and under whose supervision?"})
print(response["answer"])

This project was implemented as an application of the Expert Systems course. It was supervised by Dr. Reham El-Anany and Eng. Youssef.
